# Tutorial -- all about decorators

1. What are decorators, and how do they work?
2. Creating simple decorators
3. Creating decorators that modify inputs
4. Creating decorators that modify outputs
5. Multiple decorators
6. Decorators that take arguments
7. Improving our decorators with ... decorators!
8. (If we have time) decoratoting classes, as well

# DRY -- don't repeat yourself!

In [1]:
def a():
    return f'a!\n'

def b():
    return f'b!\n'

print(a())    
print(b())

a!

b!



In [2]:
# corporate hq says: every function's output must start and end with ----

lines = '-' * 60 + '\n'

def a():
    return f'{lines}a!\n{lines}'

def b():
    return f'{lines}b!\n{lines}'

print(a())    
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [3]:
# Option 2: instead of calling the function directly,
# I'll call it via another function. That second function
# will then insert the lines

lines = '-' * 60 + '\n'

def with_lines(func):
    return f'{lines}{func()}{lines}'

def a():
    return f'a!\n'

def b():
    return f'b!\n'

print(with_lines(a))
print(with_lines(b))  

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [4]:
# option 3: nested function -- a "closure"

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

def a():
    return f'a!\n'
with_lines_a  = with_lines(a)

def b():
    return f'b!\n'
with_lines_b = with_lines(b)

print(with_lines_a())
print(with_lines_b()) 

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [5]:
# option 4: instead of assigning to with_lines_a and with_lines_b,
# just assign to a and b!

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

def a():
    return f'a!\n'
a  = with_lines(a)

def b():
    return f'b!\n'
b = with_lines(b)

print(a())
print(b())

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------



In [ ]:
# option 5: Use Python decorator syntax

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper():
        return f'{lines}{func()}{lines}'
    return wrapper

@with_lines    # this is precisely the same as line 13's action
def a():
    return f'a!\n'
# a  = with_lines(a)

@with_lines   # this is precisely the same as line 18's actions
def b():
    return f'b!\n'
# b = with_lines(b)

print(a())
print(b())

# What is a decorator?

1. It's a function
2. That takes a function as an argument
3. That returns a function
4. That is assigned back to the original function, and runs in its place

# Who cares?

A decorator allows us to hijack a function at two points (a) definition time and (b) runtime. We can do whatever we want at either of those points in time.

1. Replace a function, perhaps if the permissions are wrong
2. Only allow a function to run under certain circumstances
3. Change/filter inputs
4. Change/filter outputs
5. Log information about the function and what it's doing

Everything we do with decorators could be done without them. But decorators give us tight syntax to do these things, and let us think at a higher level of abstraction.

In [11]:

lines = '-' * 60 + '\n'

def with_lines(func):
    def wrapper(*args, **kwargs):
        return f'{lines}{func(*args, **kwargs)}{lines}'
    return wrapper

@with_lines    # this is precisely the same as line 13's action
def a():
    return f'a!\n'
# a  = with_lines(a)

@with_lines   # this is precisely the same as line 18's actions
def b():
    return f'b!\n'
# b = with_lines(b)

@with_lines
def add(x, y):
    return f'{x} + {y} = {x+y}\n'

print(a())
print(b())
print(add(10, 30))

------------------------------------------------------------
a!
------------------------------------------------------------

------------------------------------------------------------
b!
------------------------------------------------------------

------------------------------------------------------------
10 + 30 = 40
------------------------------------------------------------



# How to write a decorator

1. Write the decorator function (what you will use with @) as a regular function that takes a single argument, which I normally call `func`. This will be the decorated function.
2. Inside of the decorator function, define an inner function, usually called `wrapper`. it should take `*args` and `**kwargs`.
3. `wrapper` can do whatever it wants -- but usually it'll end up calling the original function (available as `func`), and then storing/returning the result
4. The decorator function then needs to `return wrapper` -- we aren't invoking `wrapper`, just returning a reference to it!
5. We can apply `@decorator` to any function we want to change behavior in.

# Exercise: Timing

1. Write a decorator, called `timefunc`, that checks how long it takes for a function to run. The function will run normally, with its usual arguments, and returning its usual values, *but* we will keep track of how long each execution takes.
2. The time will be stored to a file called `timing.txt`.
3. You can get the current Unix time with `time.time()`
4. You can get the name of the function with `__name__`.
5. You can write two slow functions -- I usually write `slow_mul` and `slow_add` that just add someething like `time.sleep(random.randint(0, 3))` at the start of the function.

Practice system: 
- Practice system: https://practice.lernerpython.com/classroom/24d84164d2/
- Exercise 1: https://practice.lernerpython.com/classroom/24d84164d2/ex-1

In [5]:
import time
import random

def timefunc(func):
    def wrapper(*args, **kwargs):
        starttime = time.time()
        result = func(*args, **kwargs)
        endtime = time.time()
        with open('timing.txt', 'a') as f:
            f.write(f'{func.__name__} took {endtime - starttime:.6f} seconds\n')
        return result
    return wrapper

@timefunc
def slow_mul(x, y):
    time.sleep(random.randint(0, 3))
    return x * y

@timefunc
def slow_add(x, y):
    time.sleep(random.randint(0, 3))
    return x + y

print(slow_mul(10, 20))
print(slow_add(10, 20))

200
30


...

Exercise: Once per minute

Write a decorator, once_per_minute, that when applied to a function raises an exception (CalledTooSoonError) if invoked within 60 seconds of the previous invocation.

You can again use slow_add and slow_mul.

Exercise system: https://practice.lernerpython.com/classroom/24d84164d2/ex-2

In [23]:
import time

class CalledTooSoonError(Exception):
    pass

def once_per_five_seconds(func):
    last_call = None

    def wrapper(*args, **kwargs):
        nonlocal last_call
        if last_call is not None and time.time() - last_call < 5:
            raise CalledTooSoonError(f"{func.__name__} can only be called once per five seconds.")
        last_call = time.time()
        return func(*args, **kwargs)
    
    return wrapper

@once_per_five_seconds
def slow_mul(x, y):
    return x * y

@once_per_five_seconds
def slow_add(x, y):
    return x + y

print(slow_mul(10, 20))
print(slow_add(10, 20))
time.sleep(10)
print(slow_mul(10, 20))
print(slow_add(10, 20))
print(slow_mul(10, 20))
print(slow_add(10, 20))

200
30
200
30


CalledTooSoonError: slow_mul can only be called once per five seconds.

Exercise: only_ints

Write a decortor, only_ints, that filters the arguments to a function, and ensures that all of the arguments passed are integers. So you should be able to call

mysum(10, 15, 'hello', 20, 30, 'goodbye')    

and it'll r

On the practice system: https://practice.lernerpython.com/classroom/24d84164d2/ex-3

In [ ]:
def only_ints(func):
    def wrapper(*args):
        int_list = (arg for arg in args if isinstance(arg, int))
        
        result = func(*int_list)
        return result
    return wrapper

@only_ints
def mysum(*args):
    total = 0

    for one_number in args:
        total += one_number

    return total

mysum(10, 15, 'hello', 20, 30, 'goodbye')    


75

Exercise: only_return_ints

Write a decorator that examines the return value from the function, which we can assume is a list or other iterable. Only return the values that are ints.

Here's an awful function to play with:

def return_squares(*args):
    return [one_arg * one_arg
           for one_arg in args] + ['a', 'b', 'c']

Practice systm: https://practice.lernerpython.com/classroom/24d84164d2/ex-4

In [6]:
def only_return_ints(func):
    def wrapper(*args):
        all_args = func(*args)
        squared_list = [arg for arg in all_args if isinstance(arg, int)]

        return squared_list
    return wrapper

@only_return_ints
def return_squares(*args):
    return [one_arg * one_arg
           for one_arg in args] + ['a', 'b', 'c']

return_squares(1, 2, 3, 4, 5)

[1, 4, 9, 16, 25]